In [ ]:
# Practice training a 2 layer binary classifier with numpy
# Input layer: X shape (num_examples, num_features)
# Hidden layer: A1 = Relu(W1 @ X + B1) shape: (num_examples, hidden_dim)
# Output layer: A2 = Sigmoid(W2 @ A1 + B2) shape: (m, 1)
# Logistic loss: Loss = - (Y * log(A2) + (1 - Y) * log (1 - A2))

import numpy as np
from typing import Dict

def initialize_parameters(input_dim: int, hidden_dim: int, output_dim: int):
    W1 = np.random.rand(input_dim, hidden_dim) * 0.01 # (input_dim, hidden_dim)
    b1 = np.zeros((1, hidden_dim)) # (1, hidden_dim)
    W2 = np.random.rand(hidden_dim, output_dim) * 0.01 # (hidden_dim, output_dim)
    b2 = np.zeros((1, output_dim)) #(1, output_dim)
    parameters = {
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2
    }
    return parameters

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def forward_propagation(X, parameters):
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    b1 = parameters["b1"]
    b2 = parameters["b2"]
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }
    return A2, cache

def compute_cost(A2, Y):
    # Loss = - (Y * log(A2) + (1 - Y) * log (1 - A2))
    # Y shape (num_examples, 1)
    # A2 shape (num_examples, 1)
    eps = 1e-15
    A2_clipped = np.clip(A2, eps, 1 - eps)
    return - np.sum(Y * np.log(A2_clipped) + (1 - Y) * np.log(1 - A2_clipped)) / Y.shape[0]

def backward_propagation(parameters, cache, X, Y):
    m = X.shape[0]
    A1 = cache["A1"] # (m, hidden_dim)
    A2 = cache["A2"] # (m, output_dim)
    W1 = parameters["W1"] # (input_dim, hidden_dim)
    W2 = parameters["W2"] # (hidden_dim, output_dim)
    dZ2 = A2 - Y # (m, output_dim)
    dW2 = A1.T @ dZ2 / m # (m, hidden_dim).T @ (m, output_dim) = (hidden_dim, output_dim)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m # (1, output_dim)
    dZ1 = (dZ2 @ W2.T) * (A1 > 0) # dRelu(Z1): A1 > 0, shape (m, hidden_dim)
    dW1 = X.T @ dZ1 / m # (m, input_dim).T @ (m, hidden_dim) = (input_dim, hidden_dim)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m # (1, hidden_dim)

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }
    return grads

def update_parameters(parameters, grads, learning_rate = 1.2):
    parameters["W1"] -= learning_rate * grads["dW1"]
    parameters["W2"] -= learning_rate * grads["dW2"]
    parameters["b1"] -= learning_rate * grads["db1"]
    parameters["b2"] -= learning_rate * grads["db2"]
    return parameters

In [21]:
# Test forward
np.random.seed(48)
input_dim = 32
hidden_dim = 4
output_dim = 1
num_examples = 2
X = np.random.rand(num_examples, input_dim)
Y = np.zeros((num_examples, output_dim))
Y[0] = 1
parameters = initialize_parameters(input_dim, hidden_dim, output_dim)
A2, cache = forward_propagation(X, parameters)
cost = compute_cost(A2, Y)
print(cost)
grads = backward_propagation(parameters, cache, X, Y)
parameters = update_parameters(parameters, grads, learning_rate=0.2)
print(parameters)


0.6932488012463522
{'W1': array([[ 5.45365009e-04,  8.56552415e-04,  4.66205058e-03,
        -2.59294518e-05],
       [ 6.94306776e-03,  3.85468355e-03,  7.12979481e-03,
         6.09424811e-03],
       [ 9.38392153e-03,  9.13874817e-04,  6.17591213e-03,
         6.05862839e-03],
       [ 3.37662572e-03,  3.23037755e-03,  1.41458345e-04,
         8.22677902e-03],
       [ 7.75308892e-03,  3.79869697e-03,  4.87487036e-03,
         5.90747842e-03],
       [ 2.56652575e-03,  2.58946877e-03,  9.82192426e-03,
         4.34878226e-03],
       [ 9.57416150e-04,  3.89002971e-03,  4.24833923e-04,
         4.00787520e-04],
       [ 7.83328750e-03,  7.91367463e-03,  2.67313626e-03,
         2.32957838e-03],
       [ 2.70980906e-03,  1.00027845e-02,  7.15754800e-03,
         6.14160288e-03],
       [ 9.15724875e-03,  3.79642965e-03,  1.93469747e-03,
         7.41256531e-03],
       [ 4.10689137e-03,  7.57913962e-04,  2.50808100e-04,
         2.56937286e-03],
       [ 7.86518335e-03,  2.18641413e-0

In [26]:
def nn_model(X, Y, hidden_dim, num_iterations=1000, learning_rate=0.1, print_cost=True):
    input_dim = X.shape[1]
    output_dim = Y.shape[1]
    
    parameters = initialize_parameters(input_dim, hidden_dim, output_dim)
    costs = []
    
    for i in range(num_iterations):
        A2, cache = forward_propagation(X, parameters)
        
        cost = compute_cost(A2, Y)
        
        grads = backward_propagation(parameters, cache, X, Y)
        
        parameters = update_parameters(parameters, grads, learning_rate)
        
        if i % 100 == 0:
            costs.append(cost)
            if print_cost:
                print(f"Iteration {i:4d} | Cost: {cost:.6f}")
                
    return parameters, costs

def predict(parameters, X, threshold=0.5):
    A2, _ = forward_propagation(X, parameters)
    predictions = (A2 >= threshold).astype(int)
    return predictions, A2

num_samples = 200
X_train = np.random.randn(num_samples, input_dim)
# 简单规则生成标签：特征和大于 0 则为 1，否则为 0
Y_train = (np.sum(X_train, axis=1, keepdims=True) > 0).astype(int)

trained_params, costs = nn_model(
    X=X_train,
    Y=Y_train,
    hidden_dim=8,
    num_iterations=1000,
    learning_rate=0.2,
    print_cost=True
)

preds, probs = predict(trained_params, X_train)
accuracy = np.mean(preds == Y_train) * 100
print(f"\n accuracy: {accuracy:.2f}%")

Iteration    0 | Cost: 0.692903
Iteration  100 | Cost: 0.304157
Iteration  200 | Cost: 0.126657
Iteration  300 | Cost: 0.076239
Iteration  400 | Cost: 0.052949
Iteration  500 | Cost: 0.039747
Iteration  600 | Cost: 0.031070
Iteration  700 | Cost: 0.025029
Iteration  800 | Cost: 0.020884
Iteration  900 | Cost: 0.017837

 accuracy: 100.00%
